In [16]:
import pandas as pd

In [17]:
df = pd.read_csv("../data/rnn/IMDB Dataset.csv")

In [18]:
df.shape

(50000, 2)

In [19]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [20]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [21]:
df.drop_duplicates(inplace=True)

In [22]:
df.shape

(49582, 2)

## Pre-processing

### 1. Converting to lowercase

In [23]:
df["review"] = df["review"].str.lower()

### 2.Removing the URLs

In [24]:
import re
def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # pattern,repl,string eg - https://www.goggle.com
    return text
df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations 

In [25]:
import re
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # a-z A-Z 0-9 \s
    return text
df["review"] = df["review"].apply(remove_punctuations)   

### 4.Removing HTML tags

In [26]:
import re
def remove_html(text):
    text = re.sub(r"<.*?>", "", text) # a-z A-Z 0-9 \s
    return text
df["review"] = df["review"].apply(remove_html) 

### 5.Removing the Stopwords

In [27]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vk952\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vk952\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vk952\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [29]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words =  stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text        
df["review"] = df["review"].apply(remove_stopwords)      

In [30]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6.Stemming

In [31]:
# running -> run
# played -> play
# proterstemming

from nltk.stem import PorterStemmer

In [32]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)  
    

In [33]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7.Encoding

In [34]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [35]:
y = df["sentiment"]

In [36]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorization

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 6000)

X = tf.fit_transform(df["review"])

In [38]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4160624 stored elements and shape (49582, 6000)>

## Dataset & DAta Loader

In [39]:
from sklearn.model_selection import train_test_split

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [41]:
X_train.shape

(39665, 6000)

In [42]:
X_test.shape

(9917, 6000)

In [43]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [44]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [48]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [49]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)
                    

## Build  our RNN

In [55]:
import torch.nn as nn
import torch.optim as optim

In [56]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size , 1)

    def forward(self, x):    
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out = self.rnn(x, h0)
        # 1st value = hidden state of all the timestaps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out
        

In [58]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [70]:
import torch
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # RNN returns output and hidden state
        out, hidden = self.rnn(x)

        # Select the output from the last time step
        out = out[:, -1, :]

        # Fully connected layer
        out = self.fc(out)

        return out

In [72]:
# RNN Model
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.float().unsqueeze(1)
        yb = yb.float().reshape(-1)

        outputs = model(Xb).reshape(-1)
        outputs = torch.sigmoid(outputs)

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch = {epoch + 1}/{epochs}, Loss = {avg_loss:.4f}")
    

Epoch = 1/10, Loss = 0.4046
Epoch = 2/10, Loss = 0.2577
Epoch = 3/10, Loss = 0.2354
Epoch = 4/10, Loss = 0.2250
Epoch = 5/10, Loss = 0.2185
Epoch = 6/10, Loss = 0.2140
Epoch = 7/10, Loss = 0.2116
Epoch = 8/10, Loss = 0.2089
Epoch = 9/10, Loss = 0.2073
Epoch = 10/10, Loss = 0.2054


In [75]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted =(torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")    

accuracy = 85.38872642936371
